# OcuWeave · Cleaning review
Marchel Andrian Shevchenko · Data Sorcerers · 2026 · software v0.5.0

CPU-only technical quality screening, clinical review queues, patient-linkage evidence and separate original-note FairVLMed context review. No data deletion, annotation correction, split change or model training. Clinical and patient-identity verification remain unresolved. See `docs/protocols/cleaning_review.md` for thresholds, scope and limitations.


In [ ]:
from pathlib import Path
import os, sys, json, subprocess, hashlib, shutil
from datetime import datetime, timezone
from google.colab import drive
PROJECT=Path('/content/drive/MyDrive/OcuWeave — Datasets and experiments')
if not PROJECT.exists(): drive.mount('/content/drive')
assert PROJECT.exists(), 'Existing project Drive is required'
REVISION=os.environ.get('OCUWEAVE_REVIEW_REVISION','v0.5.0')
REPO=Path('/content/ocuweave-cleaning-review')
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/chenko-the-sorcerers/ocuweave.git',str(REPO)],check=True)
subprocess.run(['git','fetch','origin','--tags'],cwd=REPO,check=True,capture_output=True)
subprocess.run(['git','checkout','--detach',REVISION],cwd=REPO,check=True,capture_output=True)
COMMIT=subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()
STAMP=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
AUDIT=PROJECT/'02 — Audits/cleaning-review-v0.5.0'/STAMP
AUDIT.mkdir(parents=True,exist_ok=False)
ENV=dict(os.environ,OMP_NUM_THREADS='2',OPENBLAS_NUM_THREADS='2',MKL_NUM_THREADS='2',PYTHONPATH=str(REPO/'src'))
def run(command, name):
    result=subprocess.run(command,cwd=REPO,env=ENV,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
    (AUDIT/name).write_text(result.stdout)
    print(name, 'exit', result.returncode, result.stdout[-2500:])
    result.check_returncode()
def digest(path):
    h=hashlib.sha256()
    with open(path,'rb') as stream:
        for part in iter(lambda:stream.read(1024*1024),b''):h.update(part)
    return h.hexdigest()
print('Source:',REVISION,COMMIT,'CPU-only; evidence:',AUDIT)


## 1. Validate software before auditing acquired data

In [ ]:
run([sys.executable,'-m','pip','install','-e','.', '--no-deps'],'install.log')
run(['cmake','-S','.','-B','build','-DBUILD_TESTING=ON'],'cmake.log')
run(['cmake','--build','build'],'build.log')
run(['ctest','--test-dir','build','--output-on-failure'],'ctest.log')
ENV['OCUWEAVE_NATIVE_LIBRARY']=str(REPO/'build/libocuweave_native.so')
run([sys.executable,'-m','unittest','discover','-s','tests','-v'],'python-tests.log')
run([sys.executable,'scripts/audit_data.py','examples/synthetic_manifest.json'],'synthetic-manifest.log')
run([sys.executable,'-m','pip','wheel','.', '--no-deps','-w','/content/ocuweave-cleaning-wheel'],'wheel.log')
import ast, nbformat
nb=nbformat.read(REPO/'notebooks/OcuWeave_Cleaning_Review_Colab.ipynb',as_version=4)
nbformat.validate(nb)
for cell in nb.cells:
    if cell.cell_type=='code':ast.parse(cell.source)
print('Notebook schema and syntax validated')


## 2. Verify sources and preserve historical evaluation artifacts

In [ ]:
ARCHIVE=Path('/content/ocuweave-quality-otfid.zip')
ORIGINAL_ARCHIVE=PROJECT/'01 — Raw datasets/OTFID_v3/Ocular_Toxoplasmosis_Data_V3.zip'
FAIR=Path('/content/fairvlmed-quality-input')
ORIGINAL_FAIR=PROJECT/'01 — Raw datasets/FairVLMed'
if not ARCHIVE.exists():shutil.copy2(ORIGINAL_ARCHIVE,ARCHIVE)
for relative in ('Dataset/dataset.zip','ReadMe/data_summary.csv','download_manifest.json'):
    target=FAIR/relative
    if not target.exists():
        target.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(ORIGINAL_FAIR/relative,target)
MANIFEST=PROJECT/'02 — Audits/otfid-v3-binary-v2-seed42-v0.2.0/manifest.json'
PILOT=PROJECT/'04 — Runs/v0.2.0/pilot-seed42-20epochs'
EXPECTED={MANIFEST:'b8cc4f42779d57ad5ad0fa18704730edaf62650d5594996ffab1cc6d25cb980f',
    PILOT/'query-test.json':'42a1a5aac443816cb4ed08c87411249f03ef8e283851674c389ca650b90d8f42',
    PILOT/'fcn-test.json':'0a94ce5aa157d0600ea8ea41a24797e4f3d3b38a8c3887877145f8dc06191f2c'}
for path,value in EXPECTED.items():assert digest(path)==value, 'Historical artifact changed: '+path.name
PROTECTED=[MANIFEST,*PILOT.glob('*-test.json'),*PILOT.glob('*/best.pt'),ORIGINAL_ARCHIVE,
    ORIGINAL_FAIR/'Dataset/dataset.zip',ORIGINAL_FAIR/'ReadMe/data_summary.csv']
BEFORE={str(p.relative_to(PROJECT)):digest(p) for p in PROTECTED}
(AUDIT/'protected-before.json').write_text(json.dumps(BEFORE,indent=2))
print('Verified and protected',len(BEFORE),'original artifacts')


## 3. Run separate cohort reviews
Quality thresholds and previews use development images only. All-split numerical image fingerprints are permitted solely for duplicate/leakage integrity screening. No test masks, test-note analysis or model evaluation. Original notes remain private; exported note queues contain offsets and flags, not raw text.


In [ ]:
run([sys.executable,'scripts/review_cleaning.py','--archive',str(ARCHIVE),'--manifest',str(MANIFEST),
    '--fairvlmed-root',str(FAIR),'--output',str(AUDIT/'review')],'cleaning.log')
AFTER={str(p.relative_to(PROJECT)):digest(p) for p in PROTECTED}
assert BEFORE==AFTER, 'Protected artifacts changed'
(AUDIT/'protected-after.json').write_text(json.dumps(AFTER,indent=2))
REPORT=json.loads((AUDIT/'review/summary.json').read_text())
print(json.dumps(REPORT,indent=2))


## 4. Inspect development-only examples and unresolved gates

In [ ]:
from IPython.display import display, Image as DisplayImage, Markdown
for cohort in ('otfid','fairvlmed'):
    display(Markdown('### '+cohort+' · selected technical review candidates and controls'))
    for figure in sorted((AUDIT/'review'/cohort).glob('quality-example-*.png')):
        display(DisplayImage(filename=str(figure)))
    linkage=json.loads((AUDIT/'review'/cohort/'patient-linkage.json').read_text())
    print(json.dumps(linkage,indent=2))
display(Markdown('**Still unresolved:** clinical image gradability, lesion-boundary correctness/completeness, verified patient/eye/visit mapping, original-note semantic and temporal leakage. No record has been removed or used for new training.'))


## 5. Archive evidence and documentation

In [ ]:
SNAPSHOT=PROJECT/'05 — Research documentation/v0.5.0'/STAMP
run([sys.executable,'scripts/archive_research_record.py','--project',str(PROJECT),'--output',str(SNAPSHOT)],'archive.log')
print('Audit:',AUDIT,'Documentation:',SNAPSHOT)
